# RL generator: GRPO training on Colab (rl-generator SEC-211)

Trains one mode's LoRA adapter (`train.py`) and writes a batch from it (`sample.py`). This notebook only
mounts Drive, installs, sets the environment and calls the repository's scripts; the training logic is in
reviewed code. See `planning/specs/rl-generator/design-low-level-generator.md` §8.

Before running: upload `seeds_<mode>.jsonl` (mode in lower case), `novelty_bank.npy` (+ `.json`) and `judge_cache.sqlite` from the
Mac's `backend/data/security/alignment/local/generated/` to `MyDrive/webagent-rl/`, and add `TOGETHER_API_KEY`
in Colab's secrets panel. Commit this notebook with outputs cleared. After the run, download the batch file
and the judge cache, then delete the batch file from Drive.

In [ ]:
# Parameters (set before running)
REF = "v3.14.0"                      # release tag with the RL generator tooling
MODE = "H"                           # "H" or "L"
BASE = "Qwen/Qwen2.5-1.5B-Instruct"  # Qwen/Qwen2.5-0.5B-Instruct for the 20-step pilot
MAX_STEPS = 20                       # from the user's compute-unit cap and the pilot's units per step
CREDIT_LEFT = 0.0                    # Together credit left ($), for the judge's stop rule
JUDGE_MODEL = ""                     # the judge chosen in task G6
PRICE_IN, PRICE_OUT = 0.0, 0.0       # the judge model's $ per million input / output tokens (G6)
N_ROWS = 500                         # rows wanted from sample.py
EXTRA_SAMPLE_ARGS = []               # e.g. ["--heldout", "--step", "300", "--sample-seed", "11"]

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
RL = "/content/drive/MyDrive/webagent-rl"
os.makedirs(f"{RL}/{MODE}", exist_ok=True)

In [ ]:
%%bash -s "$REF"
set -e
rm -rf /content/WebAgent
git clone --depth 1 --branch "$1" https://github.com/Positron00/WebAgent.git /content/WebAgent
cd /content/WebAgent
git lfs install --local
git lfs pull --include "backend/models/security/**"
grep -v '^fsspec==' backend/constraints.txt > /tmp/constraints-train.txt
pip install -q -r backend/requirements.txt -r backend/requirements-train.txt -c /tmp/constraints-train.txt

In [ ]:
import shutil
GEN = "/content/WebAgent/backend/data/security/alignment/local/generated"
os.makedirs(GEN, exist_ok=True)
for name in (f"seeds_{MODE.lower()}.jsonl", "novelty_bank.npy", "novelty_bank.json", "judge_cache.sqlite"):
    if os.path.exists(f"{RL}/{name}"):
        shutil.copy(f"{RL}/{name}", f"{GEN}/{name}")
print(sorted(os.listdir(GEN)))

In [ ]:
from google.colab import userdata
os.environ["TOGETHER_API_KEY"] = userdata.get("TOGETHER_API_KEY")  # never printed
os.environ["WEBAGENT_ENV"] = "dev"
for name in ("LANGSMITH_TRACING_ENABLED", "LANGSMITH_TRACING", "LANGCHAIN_TRACING_V2"):
    os.environ[name] = "false"
os.chdir("/content/WebAgent/backend")

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/rl_generator/train.py", "--mode", MODE, "--base", BASE,
                "--seeds", f"{GEN}/seeds_{MODE.lower()}.jsonl", "--bank", f"{GEN}/novelty_bank.npy",
                "--target", "models/security/alignment", "--judge-model", JUDGE_MODEL,
                "--ckpt-dir", f"{RL}/{MODE}", "--max-steps", str(MAX_STEPS),
                "--credit-left", str(CREDIT_LEFT),
                "--price-input", str(PRICE_IN), "--price-output", str(PRICE_OUT)], check=True)

In [ ]:
subprocess.run([sys.executable, "scripts/rl_generator/sample.py", "--mode", MODE, "--base", BASE,
                "--seeds", f"{GEN}/seeds_{MODE.lower()}.jsonl", "--bank", f"{GEN}/novelty_bank.npy",
                "--target", "models/security/alignment", "--judge-model", JUDGE_MODEL,
                "--ckpt-dir", f"{RL}/{MODE}", "--n", str(N_ROWS),
                "--credit-left", str(CREDIT_LEFT),
                "--price-input", str(PRICE_IN), "--price-output", str(PRICE_OUT), *EXTRA_SAMPLE_ARGS], check=True)

In [ ]:
# Copy the batch file(s) and the judge cache back to Drive for download to the Mac.
for name in os.listdir(GEN):
    if name.startswith(f"rl_gen_{MODE.lower()}_") or name == "judge_cache.sqlite":
        shutil.copy(f"{GEN}/{name}", f"{RL}/{name}")
print("copied")